# Phase P2: Batch Size Probing & Diagnostic Sweeps
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkTinkerGreen/train_with_reversibility/blob/main/notebooks/01_batch_probe.ipynb)


In [ ]:
import os, sys, json
SMOKE = os.environ.get("REVLLM_SMOKE", "0") == "1"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/train_with_reversibility' if os.path.exists('/content/train_with_reversibility') else '.')


In [ ]:
# Run deterministic batch size probe algorithm (exponential + binary search)
from scripts.probe import probe_arch_max_batch, run_diagnostic_sweeps

archs = ["baseline", "midpoint"] if SMOKE else ["baseline", "midpoint", "midpoint_theta", "leapfrog"]
b_max_map = {}
b_safe_map = {}
all_probes = []

for arch in archs:
    b_max, b_safe, probes = probe_arch_max_batch(arch)
    b_max_map[arch] = b_max
    b_safe_map[arch] = b_safe
    all_probes.extend(probes)

b_A = b_safe_map["baseline"]
winner_candidate = "midpoint"
b_C = b_safe_map[winner_candidate]

sweeps = run_diagnostic_sweeps(winner_candidate, b_max_map["baseline"], b_max_map[winner_candidate])

probe_output = {
    "b_A": b_A,
    "b_C": b_C,
    "b_max_map": b_max_map,
    "b_safe_map": b_safe_map,
    "probe_history": all_probes,
    "diagnostic_sweeps": sweeps,
}

os.makedirs("results", exist_ok=True)
with open("results/batch_probe.json", "w") as f:
    json.dump(probe_output, f, indent=2)

if IN_COLAB and not SMOKE:
    import shutil
    shutil.copy2("results/batch_probe.json", "/content/drive/MyDrive/revllm/probes/batch_probe.json")

print(f"Batch Probe Done! Baseline B_A={b_A}, Reversible B_C={b_C}")
